# LeetCode 3910 — Count Connected Subgraphs with Even Node Sum bằng Python

- Link: https://leetcode.com/problems/count-connected-subgraphs-with-even-node-sum/
- Độ khó: **Hard**
- Chủ đề: Bitmask, duyệt tập con, đồ thị con cảm sinh (induced subgraph), BFS/DFS trên bitmask, quy hoạch động trên tập con, liệt kê tập liên thông

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def evenSumSubgraphs(self, nums: list[int], edges: list[list[int]]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra tín hiệu **`n ≤ 13`** ⇒ duyệt toàn bộ `2^n` tập con là khả thi.
2. Biểu diễn tập đỉnh bằng **số nguyên bitmask**; dùng `adj_mask[v]` (bitmask láng giềng) để "loang" (flood fill) chỉ bằng phép `&`, `|`.
3. Tính tổng chẵn/lẻ nhanh bằng `popcount(mask & ones)`.
4. Viết **DP trên tập con** `conn[mask]` dựa trên tính chất "mọi đồ thị liên thông có một đỉnh không phải đỉnh cắt".
5. Biết kỹ thuật **chỉ liệt kê các tập liên thông** (mỗi tập đúng một lần) bằng quay lui với tập "cấm".

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho đồ thị vô hướng `n` đỉnh, đỉnh `i` có giá trị `nums[i] ∈ {0, 1}`. Với mỗi **tập đỉnh khác rỗng** `s`, xét **đồ thị con cảm sinh**: giữ các đỉnh trong `s` và chỉ những cạnh có **cả hai đầu** trong `s`. Đếm số tập `s` sao cho:

- đồ thị con cảm sinh **liên thông**;
- tổng giá trị các đỉnh trong `s` là **số chẵn**.

**Ví dụ 1:** `nums = [1,0,1]`, `edges = [[0,1],[1,2]]` → `2` (tập `{1}` và `{0,1,2}`).

**Ví dụ 2:** `nums = [1]`, `edges = []` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 13`, `nums[i] ∈ {0,1}`, `0 ≤ edges.length ≤ n(n-1)/2`, `0 ≤ u < v < n`, cạnh phân biệt.

In [1]:
from collections import deque
import random, time

examples = [([1, 0, 1], [[0, 1], [1, 2]], 2), ([1], [], 0)]
for nums, edges, exp in examples:
    print(nums, edges, "-> kỳ vọng", exp)

[1, 0, 1] [[0, 1], [1, 2]] -> kỳ vọng 2
[1] [] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Bitmask biểu diễn tập con

Tập `s ⊆ {0..n-1}` ↔ số nguyên `mask`, bit `i` bật ⇔ `i ∈ s`.

| Thao tác | Code |
|---|---|
| `i ∈ s`? | `mask >> i & 1` |
| thêm `i` | `mask | (1 << i)` |
| bỏ `i` | `mask & ~(1 << i)` |
| bit thấp nhất | `mask & -mask` |
| chỉ số bit thấp nhất | `(mask & -mask).bit_length() - 1` |
| số phần tử | `bin(mask).count('1')` hoặc `mask.bit_count()` (Python ≥ 3.10) |

In [2]:
mask = 0b10110           # tập {1, 2, 4}
print([i for i in range(5) if mask >> i & 1])
low = mask & -mask
print("bit thấp nhất:", bin(low), "-> đỉnh", low.bit_length() - 1)
print("số phần tử:", bin(mask).count('1'))

[1, 2, 4]
bit thấp nhất: 0b10 -> đỉnh 1
số phần tử: 3


### 3.2. Bitmask láng giềng và "loang" bằng phép bit

`adj[v]` = bitmask các láng giềng của `v`. Để tìm mọi đỉnh tới được từ `start` **trong tập `sub`**: bắt đầu `seen = start`, lặp: lấy một đỉnh chưa mở rộng, `seen |= adj[v] & sub`. Mỗi đỉnh được mở rộng một lần ⇒ `O(n)` thao tác bit.

In [3]:
def build_adj_mask(n, edges):
    adj = [0] * n
    for u, v in edges:
        adj[u] |= 1 << v
        adj[v] |= 1 << u
    return adj

def reach_within(adj, sub, start_bit):
    seen = start_bit
    todo = start_bit
    while todo:
        b = todo & -todo
        todo ^= b
        v = b.bit_length() - 1
        new = adj[v] & sub & ~seen
        seen |= new
        todo |= new
    return seen

adj = build_adj_mask(3, [[0, 1], [1, 2]])
print([bin(a) for a in adj])
print("từ đỉnh 0 trong {0,2}:", bin(reach_within(adj, 0b101, 0b001)))
print("từ đỉnh 0 trong {0,1,2}:", bin(reach_within(adj, 0b111, 0b001)))

['0b10', '0b101', '0b10']
từ đỉnh 0 trong {0,2}: 0b1
từ đỉnh 0 trong {0,1,2}: 0b111


### 3.3. Tính chẵn lẻ của tổng bằng popcount

Vì `nums[i] ∈ {0,1}`, tổng trên tập `mask` = số đỉnh có giá trị 1 trong `mask` = `popcount(mask & ones)` với `ones` là bitmask các đỉnh có `nums[i] = 1`.

In [4]:
nums = [1, 0, 1]
ones = sum(1 << i for i, x in enumerate(nums) if x)
for mask in range(1, 8):
    print(bin(mask), "tổng =", bin(mask & ones).count('1'))

0b1 tổng = 1
0b10 tổng = 0
0b11 tổng = 1
0b100 tổng = 1
0b101 tổng = 2
0b110 tổng = 1
0b111 tổng = 2


### 3.4. Disjoint Set Union (DSU)

DSU gộp các đỉnh theo cạnh; đồ thị liên thông ⇔ cuối cùng chỉ còn 1 thành phần. Dùng ở cách baseline.

In [5]:
def dsu_components(vertices, edge_list):
    parent = {v: v for v in vertices}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    comps = len(vertices)
    for u, v in edge_list:
        ru, rv = find(u), find(v)
        if ru != rv:
            parent[ru] = rv
            comps -= 1
    return comps

print(dsu_components([0, 1, 2], [(0, 1)]), dsu_components([0, 1, 2], [(0, 1), (1, 2)]))

2 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Liệt kê tay ví dụ 1 (`nums = [1,0,1]`, đường thẳng `0 - 1 - 2`):

| tập | liên thông? | tổng | đếm? |
|---|---|---|---|
| {0} | có | 1 | không |
| {1} | có | 0 | **có** |
| {2} | có | 1 | không |
| {0,1} | có | 1 | không |
| {0,2} | **không** (cạnh 0-2 không tồn tại) | 2 | không |
| {1,2} | có | 1 | không |
| {0,1,2} | có | 2 | **có** |

Đáp án `2`. Lưu ý `{0,2}`: dù trong đồ thị gốc 0 và 2 nối qua 1, nhưng **đồ thị cảm sinh** bỏ mất đỉnh 1 nên không liên thông.

Với `n ≤ 13`, có tối đa `2^13 - 1 = 8191` tập — ta hoàn toàn có thể kiểm tra từng tập.

## 5. Cách 1 — Brute Force: duyệt tập con + dựng đồ thị cảm sinh + DSU

**Ý tưởng:** với mỗi `mask`, lấy danh sách đỉnh, lọc cạnh có hai đầu trong tập, đếm thành phần bằng DSU. Nếu 1 thành phần và tổng chẵn thì đếm.

In [6]:
def brute(nums, edges):
    n = len(nums)
    ans = 0
    for mask in range(1, 1 << n):
        verts = [i for i in range(n) if mask >> i & 1]
        if sum(nums[i] for i in verts) % 2:
            continue
        sub_edges = [(u, v) for u, v in edges if mask >> u & 1 and mask >> v & 1]
        if dsu_components(verts, sub_edges) == 1:
            ans += 1
    return ans

for nums, edges, exp in examples:
    print(brute(nums, edges), "kỳ vọng", exp)

2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** `O(2^n · (n + m))` với `m ≤ 78` — khoảng `8191 · 91 ≈ 7,5·10^5` thao tác, **đã đủ nhanh** cho ràng buộc. Nhưng hằng số lớn (list, dict, lọc cạnh). Các cách sau tối ưu bằng phép bit và cho thêm góc nhìn DP/liệt kê.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — Bitmask hóa toàn bộ:** thay danh sách cạnh bằng `adj[v]` (bitmask). Liên thông của tập `sub` ⇔ loang từ một đỉnh của `sub` (chỉ đi trong `sub`) phủ hết `sub`. Tổng chẵn ⇔ `popcount(sub & ones)` chẵn.

**Insight 2 — DP trên tập con:** mọi đồ thị liên thông có ít nhất 2 đỉnh đều có một đỉnh **không phải đỉnh cắt** (ví dụ một lá của cây khung). Do đó:

> `conn[mask]` đúng ⇔ `mask` chỉ có 1 đỉnh, **hoặc** tồn tại `v ∈ mask` sao cho `conn[mask \ {v}]` đúng **và** `v` kề ít nhất một đỉnh trong `mask \ {v}`.

**Insight 3 — Chỉ liệt kê tập liên thông:** thay vì duyệt cả `2^n` tập, mở rộng dần một tập liên thông từ đỉnh gốc nhỏ nhất, mỗi lần thêm một láng giềng. Dùng tập "cấm" để mỗi tập được sinh **đúng một lần**. Số bước tỉ lệ với số tập liên thông (có thể ít hơn nhiều `2^n` khi đồ thị thưa).

## 7. Các cách tiếp theo

- **Cách 2:** duyệt tập con + loang bitmask + popcount — `O(2^n · n)`, lời giải nộp chính.
- **Cách 3:** DP trên tập con `conn[mask]` — `O(2^n · n)`.
- **Cách 4:** quay lui liệt kê **chỉ** các tập liên thông — `O(#tập liên thông · n)`.

### 7.1. Cách 2 — Duyệt tập con + loang bằng bitmask

**Giải thích từng đoạn:**

1. Dựng `adj` (bitmask láng giềng) và `ones`.
2. Với mỗi `sub` từ `1` tới `2^n - 1`: bỏ qua nếu `popcount(sub & ones)` lẻ.
3. Loang từ bit thấp nhất của `sub` trong phạm vi `sub`; nếu phủ hết `sub` thì đếm.

In [7]:
def bitmask_flood(nums, edges):
    n = len(nums)
    adj = build_adj_mask(n, edges)
    ones = sum(1 << i for i, x in enumerate(nums) if x)
    ans = 0
    for sub in range(1, 1 << n):
        if bin(sub & ones).count('1') & 1:
            continue
        if reach_within(adj, sub, sub & -sub) == sub:
            ans += 1
    return ans

for nums, edges, exp in examples:
    print(bitmask_flood(nums, edges), "kỳ vọng", exp)

2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** `O(2^n · n)` thao tác bit, bộ nhớ `O(n)`.

### 7.2. Cách 3 — DP trên tập con

**Giải thích:** duyệt `mask` tăng dần (mọi tập con thật sự của `mask` đều nhỏ hơn `mask`, nên đã được tính). Với `mask` có ≥ 2 bit, thử bỏ từng đỉnh `v`.

Tối ưu nhỏ: chỉ cần **một** `v` thỏa là dừng (`break`).

In [8]:
def subset_dp(nums, edges):
    n = len(nums)
    adj = build_adj_mask(n, edges)
    ones = sum(1 << i for i, x in enumerate(nums) if x)
    full = 1 << n
    conn = [False] * full
    ans = 0
    for mask in range(1, full):
        if mask & (mask - 1) == 0:          # đúng 1 bit
            conn[mask] = True
        else:
            m = mask
            while m:
                b = m & -m
                m ^= b
                rest = mask ^ b
                if conn[rest] and adj[b.bit_length() - 1] & rest:
                    conn[mask] = True
                    break
        if conn[mask] and bin(mask & ones).count('1') % 2 == 0:
            ans += 1
    return ans

for nums, edges, exp in examples:
    print(subset_dp(nums, edges), "kỳ vọng", exp)

2 kỳ vọng 2
0 kỳ vọng 0


In [9]:
class SolutionDP:
    def evenSumSubgraphs(self, nums: list[int], edges: list[list[int]]) -> int:
        return subset_dp(nums, edges)

print([SolutionDP().evenSumSubgraphs(a, e) for a, e, _ in examples])

[2, 0]


**Độ phức tạp:** `O(2^n · n)` thời gian, `O(2^n)` bộ nhớ (mảng `conn`). Lợi ích: `conn` dùng lại được cho nhiều câu hỏi khác (ví dụ đếm theo tổng bất kỳ).

### 7.3. Cách 4 — Chỉ liệt kê các tập liên thông (quay lui với tập cấm)

**Ý tưởng:** mỗi tập liên thông có một đỉnh nhỏ nhất `r` (gốc). Với gốc `r`, ta sinh mọi tập liên thông chứa `r` và chỉ gồm đỉnh `≥ r`:

- trạng thái `(S, F, B)`: `S` là tập hiện tại (liên thông), `F` là các ứng viên có thể thêm (láng giềng của `S`, chưa bị cấm), `B` là tập **cấm**;
- đếm `S`; rồi với từng `u` trong `F` (lần lượt): đệ quy với `S ∪ {u}`, ứng viên mới = (phần còn lại của `F`) ∪ (láng giềng của `u`) trừ `S ∪ {u}` và `B`; sau khi quay về thì **cấm `u`** cho các nhánh sau.

Cấm `u` sau khi đã xét nhánh "có `u`" đảm bảo nhánh sau là "không có `u`" ⇒ mỗi tập sinh đúng một lần.

In [10]:
def enumerate_connected(nums, edges):
    n = len(nums)
    adj = build_adj_mask(n, edges)
    ones = sum(1 << i for i, x in enumerate(nums) if x)
    ans = 0

    def rec(S, F, B):
        nonlocal ans
        if bin(S & ones).count('1') % 2 == 0:
            ans += 1
        while F:
            u = F & -F
            F ^= u
            v = u.bit_length() - 1
            S2 = S | u
            rec(S2, (F | adj[v]) & ~S2 & ~B, B)
            B |= u

    for r in range(n):
        B = (1 << (r + 1)) - 1                 # cấm mọi đỉnh <= r (r đã nằm trong S)
        rec(1 << r, adj[r] & ~B, B)
    return ans

for nums, edges, exp in examples:
    print(enumerate_connected(nums, edges), "kỳ vọng", exp)

2 kỳ vọng 2
0 kỳ vọng 0


In [11]:
# Kiểm tra "mỗi tập sinh đúng một lần": thu thập danh sách tập rồi so với tập liên thông thật
def list_connected(n, edges):
    adj = build_adj_mask(n, edges); out = []
    def rec(S, F, B):
        out.append(S)
        while F:
            u = F & -F; F ^= u; S2 = S | u
            rec(S2, (F | adj[u.bit_length() - 1]) & ~S2 & ~B, B)
            B |= u
    for r in range(n):
        B = (1 << (r + 1)) - 1
        rec(1 << r, adj[r] & ~B, B)
    return out

n, e = 5, [[0, 1], [1, 2], [2, 3], [3, 4], [0, 4], [1, 3]]
gen = list_connected(n, e)
adj = build_adj_mask(n, e)
truth = [s for s in range(1, 1 << n) if reach_within(adj, s, s & -s) == s]
print("số tập sinh ra:", len(gen), "| không trùng:", len(gen) == len(set(gen)), "| khớp tập thật:", sorted(gen) == truth)

số tập sinh ra: 24 | không trùng: True | khớp tập thật: True


In [12]:
class SolutionEnum:
    def evenSumSubgraphs(self, nums: list[int], edges: list[list[int]]) -> int:
        return enumerate_connected(nums, edges)

print([SolutionEnum().evenSumSubgraphs(a, e) for a, e, _ in examples])

[2, 0]


**Độ phức tạp Cách 4:** `O(C · n)` với `C` = số tập liên thông (≤ `2^n`). Với đồ thị thưa (đường thẳng có `C = n(n+1)/2`) nhanh hơn rất nhiều so với duyệt `2^n`.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1 (`ones = 0b101`, `adj[0] = 0b010`, `adj[1] = 0b101`, `adj[2] = 0b010`).

| sub | tổng | loang từ bit thấp | phủ hết? | đếm |
|---|---|---|---|---|
| 001 | 1 | – (lẻ, bỏ qua) | – | – |
| 010 | 0 | {1} | ✔ | 1 |
| 011 | 1 | – | – | – |
| 100 | 1 | – | – | – |
| 101 | 2 | {0} (không qua được 1) | ✘ | – |
| 110 | 1 | – | – | – |
| 111 | 2 | {0,1,2} | ✔ | 2 |

In [13]:
def trace_flood(nums, edges):
    n = len(nums)
    adj = build_adj_mask(n, edges)
    ones = sum(1 << i for i, x in enumerate(nums) if x)
    ans = 0
    for sub in range(1, 1 << n):
        s = bin(sub & ones).count('1')
        if s & 1:
            print(f"sub={sub:0{n}b} tổng={s} -> lẻ, bỏ qua")
            continue
        seen = reach_within(adj, sub, sub & -sub)
        ok = seen == sub
        ans += ok
        print(f"sub={sub:0{n}b} tổng={s} -> loang được {seen:0{n}b} -> {'liên thông' if ok else 'không liên thông'} | ans={ans}")

trace_flood([1, 0, 1], [[0, 1], [1, 2]])

sub=001 tổng=1 -> lẻ, bỏ qua
sub=010 tổng=0 -> loang được 010 -> liên thông | ans=1
sub=011 tổng=1 -> lẻ, bỏ qua
sub=100 tổng=1 -> lẻ, bỏ qua
sub=101 tổng=2 -> loang được 001 -> không liên thông | ans=1
sub=110 tổng=1 -> lẻ, bỏ qua
sub=111 tổng=2 -> loang được 111 -> liên thông | ans=2


In [14]:
# Trace DP conn[mask] cho đồ thị tam giác thiếu một cạnh: 0-1, 1-2 (n = 3)
n, e = 3, [[0, 1], [1, 2]]
adj = build_adj_mask(n, e)
conn = [False] * (1 << n)
for mask in range(1, 1 << n):
    if mask & (mask - 1) == 0:
        conn[mask] = True; why = "1 đỉnh"
    else:
        why = "không có v phù hợp"
        for v in range(n):
            if mask >> v & 1:
                rest = mask ^ (1 << v)
                if conn[rest] and adj[v] & rest:
                    conn[mask] = True; why = f"bỏ v={v}: rest={rest:03b} liên thông và v kề rest"; break
    print(f"mask={mask:03b} conn={conn[mask]} ({why})")

mask=001 conn=True (1 đỉnh)
mask=010 conn=True (1 đỉnh)
mask=011 conn=True (bỏ v=0: rest=010 liên thông và v kề rest)
mask=100 conn=True (1 đỉnh)
mask=101 conn=False (không có v phù hợp)
mask=110 conn=True (bỏ v=1: rest=100 liên thông và v kề rest)
mask=111 conn=True (bỏ v=0: rest=110 liên thông và v kề rest)


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def evenSumSubgraphs(self, nums: list[int], edges: list[list[int]]) -> int:
        n = len(nums)
        adj = [0] * n
        for u, v in edges:
            adj[u] |= 1 << v
            adj[v] |= 1 << u
        ones = 0
        for i, x in enumerate(nums):
            if x:
                ones |= 1 << i
        ans = 0
        for sub in range(1, 1 << n):
            if bin(sub & ones).count('1') & 1:
                continue                      # tổng lẻ
            seen = todo = sub & -sub
            while todo:
                b = todo & -todo
                todo ^= b
                new = adj[b.bit_length() - 1] & sub & ~seen
                seen |= new
                todo |= new
            if seen == sub:
                ans += 1
        return ans

sol = Solution()
for nums, edges, exp in examples:
    got = sol.evenSumSubgraphs(nums, edges)
    print(got, "✔" if got == exp else "✘")

2 ✔
0 ✔


## 10. Vì sao thuật toán đúng

**Cách 2.** Phép loang chỉ đi qua cạnh `v → w` khi `w ∈ sub` (lọc `& sub`), tức chỉ dùng cạnh của **đồ thị cảm sinh**. Mỗi đỉnh được thêm vào `todo` đúng một lần (vì `& ~seen`), nên vòng lặp dừng và `seen` = thành phần liên thông chứa đỉnh xuất phát. Tập liên thông ⇔ thành phần đó bằng cả `sub`. Tổng giá trị = số đỉnh giá trị 1 = `popcount(sub & ones)`.

**Cách 3.** (⇐) Nếu `rest = mask \ {v}` liên thông và `v` kề một đỉnh của `rest` thì thêm `v` vẫn liên thông. (⇒) Nếu `mask` liên thông, ≥ 2 đỉnh, lấy một cây khung; một lá `v` của cây khung có `mask \ {v}` vẫn liên thông (cây khung bỏ lá vẫn là cây) và `v` kề cha của nó.

**Cách 4.** Mỗi tập liên thông `T` có gốc `r = min T`. Trong cây quay lui gốc `r`, mỗi lần gặp ứng viên `u` ta tách thành hai nhánh rời nhau "có `u`" / "không có `u`" (cấm `u`), nên mỗi tập sinh nhiều nhất một lần. Mỗi tập liên thông đều được sinh: thêm các đỉnh theo thứ tự BFS từ `r` trong `T`, mỗi đỉnh khi đó là láng giềng của tập hiện tại nên luôn nằm trong ứng viên hoặc sẽ được xét ở nhánh tương ứng.

## 11. Phân tích Time / Space Complexity

Đặt `n` ≤ 13 đỉnh, `m` cạnh, `C` = số tập liên thông.

| Cách | Time | Space |
|---|---|---|
| 1. Tập con + DSU | `O(2^n (n + m))` | `O(n + m)` |
| 2. Tập con + loang bit | `O(2^n · n)` | `O(n)` |
| 3. DP trên tập con | `O(2^n · n)` | `O(2^n)` |
| 4. Liệt kê tập liên thông | `O(C · n)` | `O(n)` (đệ quy sâu ≤ n) |

Với `n = 13`: `2^13 · 13 ≈ 10^5` thao tác bit — tức thì.

## 12. Bộ test

In [16]:
def run_all(nums, edges):
    return [brute(nums, edges), bitmask_flood(nums, edges), subset_dp(nums, edges),
            enumerate_connected(nums, edges), Solution().evenSumSubgraphs(nums, edges)]

for nums, edges, exp in examples:
    assert all(v == exp for v in run_all(nums, edges))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [17]:
# 12.2 ca biên (giá trị kỳ vọng tự tính, kiểm chứng bằng brute)
cases = [
    ([0], [], 1),                                     # một đỉnh giá trị 0
    ([0, 0, 0], [], 3),                               # không cạnh: chỉ các tập 1 đỉnh
    ([1, 1], [[0, 1]], 1),                            # {0,1} tổng 2
    ([0, 0, 0], [[0, 1], [1, 2], [0, 2]], 7),         # tam giác toàn 0: mọi tập khác rỗng
    ([1] * 4, [[0, 1], [1, 2], [2, 3]], 4),           # đường 4 đỉnh toàn 1: 3 đoạn độ dài 2 + 1 đoạn độ dài 4
]
for nums, edges, exp in cases:
    r = run_all(nums, edges)
    assert all(v == exp for v in r), (nums, edges, r)
print("Ca biên: OK")

Ca biên: OK


In [18]:
# 12.3 phản ví dụ: dùng liên thông của đồ thị GỐC thay vì đồ thị cảm sinh
def wrong_original_conn(nums, edges):
    n = len(nums); adj = build_adj_mask(n, edges); full = (1 << n) - 1
    ans = 0
    for sub in range(1, 1 << n):
        if sum(nums[i] for i in range(n) if sub >> i & 1) % 2: continue
        # SAI: loang trong toàn đồ thị, chỉ kiểm tra sub nằm trong cùng thành phần
        if reach_within(adj, full, sub & -sub) & sub == sub:
            ans += 1
    return ans

print("sai:", wrong_original_conn([1, 0, 1], [[0, 1], [1, 2]]), "| đúng:", Solution().evenSumSubgraphs([1, 0, 1], [[0, 1], [1, 2]]))

sai: 3 | đúng: 2


In [19]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3910)
for _ in range(300):
    n = rng.randint(1, 9)
    nums = [rng.randint(0, 1) for _ in range(n)]
    all_e = [[u, v] for u in range(n) for v in range(u + 1, n)]
    edges = [e for e in all_e if rng.random() < rng.choice([0.2, 0.4, 0.7])]
    r = run_all(nums, edges)
    assert len(set(r)) == 1, (nums, edges, r)
print("300 test ngẫu nhiên: OK")

300 test ngẫu nhiên: OK


In [20]:
# 12.5 test lớn nhất: n = 13, đồ thị đầy đủ và đồ thị thưa
n = 13
nums = [i % 2 for i in range(n)]
full_edges = [[u, v] for u in range(n) for v in range(u + 1, n)]
t0 = time.time(); a = Solution().evenSumSubgraphs(nums, full_edges); t1 = time.time()
# đồ thị đầy đủ: mọi tập khác rỗng đều liên thông -> số tập có số đỉnh-1 chẵn
from math import comb
ones_cnt, zeros_cnt = sum(nums), n - sum(nums)
expected = sum(comb(ones_cnt, j) for j in range(0, ones_cnt + 1, 2)) * 2 ** zeros_cnt - 1
print("đồ thị đầy đủ:", a, "| công thức:", expected, "| khớp:", a == expected, "| thời gian:", round(t1 - t0, 3), "s")
path_edges = [[i, i + 1] for i in range(n - 1)]
print("đường thẳng 13 đỉnh:", Solution().evenSumSubgraphs(nums, path_edges), "| liệt kê:", enumerate_connected(nums, path_edges))

đồ thị đầy đủ: 4095 | công thức: 4095 | khớp: True | thời gian: 0.008 s
đường thẳng 13 đỉnh: 43 | liệt kê: 43


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra liên thông trên đồ thị gốc (quên "cảm sinh")

Đã minh họa ở 12.3: `{0, 2}` trong đường `0-1-2` cùng thành phần của đồ thị gốc nhưng **không** liên thông khi bỏ đỉnh 1. **Sửa:** khi loang phải `& sub`.

### Lỗi 2 — Đếm cả tập rỗng

In [21]:
def wrong_include_empty(nums, edges):
    n = len(nums); adj = build_adj_mask(n, edges)
    ones = sum(1 << i for i, x in enumerate(nums) if x)
    ans = 0
    for sub in range(0, 1 << n):                       # SAI: bắt đầu từ 0
        if bin(sub & ones).count('1') % 2: continue
        if reach_within(adj, sub, sub & -sub) == sub:  # tập rỗng: seen = 0 == sub -> đếm nhầm
            ans += 1
    return ans

print("sai:", wrong_include_empty([1], []), "| đúng:", Solution().evenSumSubgraphs([1], []))

sai: 1 | đúng: 0


**Sửa:** duyệt `sub` từ `1`.

### Lỗi 3 — DP trên tập con chỉ thử bỏ **đỉnh lớn nhất**

Bỏ đỉnh lớn nhất có thể làm mất liên thông (nếu nó là đỉnh cắt), nên DP sai. Phải thử **mọi** đỉnh (hoặc lập luận khác).

In [22]:
def wrong_dp_highest(nums, edges):
    n = len(nums); adj = build_adj_mask(n, edges)
    ones = sum(1 << i for i, x in enumerate(nums) if x)
    conn = [False] * (1 << n); ans = 0
    for mask in range(1, 1 << n):
        if mask & (mask - 1) == 0:
            conn[mask] = True
        else:
            v = mask.bit_length() - 1                   # SAI: chỉ thử đỉnh lớn nhất
            rest = mask ^ (1 << v)
            conn[mask] = conn[rest] and bool(adj[v] & rest)
        if conn[mask] and bin(mask & ones).count('1') % 2 == 0:
            ans += 1
    return ans

nums, edges = [0, 0, 0], [[0, 2], [1, 2]]       # đường 0 - 2 - 1: đỉnh 2 là đỉnh cắt
print("sai:", wrong_dp_highest(nums, edges), "| đúng:", Solution().evenSumSubgraphs(nums, edges))

sai: 5 | đúng: 6


### Lỗi 4 — Đệ quy DFS bằng closure định nghĩa lại trong vòng lặp + quên reset

Viết `def dfs` bên trong vòng lặp với `nonlocal vis` dễ quên khởi tạo `vis` cho từng tập, khiến kết quả của tập trước "rò" sang tập sau. Dùng hàm thuần (nhận `sub`, trả `seen`) như `reach_within` sẽ an toàn hơn.

In [23]:
def wrong_leaky(nums, edges):
    n = len(nums); adj = build_adj_mask(n, edges)
    ones = sum(1 << i for i, x in enumerate(nums) if x)
    seen = 0                                           # SAI: khởi tạo một lần ngoài vòng lặp
    ans = 0
    for sub in range(1, 1 << n):
        if bin(sub & ones).count('1') & 1: continue
        todo = sub & -sub; seen |= todo
        while todo:
            b = todo & -todo; todo ^= b
            new = adj[b.bit_length() - 1] & sub & ~seen
            seen |= new; todo |= new
        if seen & sub == sub:
            ans += 1
    return ans

print("sai:", wrong_leaky([1, 0, 1], [[0, 1], [1, 2]]), "| đúng:", Solution().evenSumSubgraphs([1, 0, 1], [[0, 1], [1, 2]]))

sai: 1 | đúng: 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **`n ≤ 15–20`** trên đồ thị/tập hợp ⇒ nghĩ ngay tới **duyệt bitmask** `2^n`.
- **Đồ thị con cảm sinh** ⇒ khi duyệt cạnh/láng giềng luôn `& sub`.
- **Giá trị 0/1, hỏi chẵn lẻ** ⇒ `popcount(mask & ones) & 1`.
- **Tính chất "liên thông" trên mọi tập con** ⇒ DP `conn[mask]` hoặc loang bit; nếu cần đếm số tập liên thông với đồ thị thưa, **liệt kê trực tiếp** nhanh hơn.
- Dùng `mask & -mask` để lấy đỉnh bắt đầu, tránh vòng `for i in range(n)`.

## 15. Biến thể của bài

1. **Tổng chia hết cho `k`** (giá trị bất kỳ): thay popcount bằng tổng `sum_mask[mask]` tính bằng DP `sum[mask] = sum[mask ^ low] + val[low]`.
2. **Đếm tập liên thông theo kích thước**: dùng `conn[mask]` rồi nhóm theo `popcount`.
3. **Cây con liên thông trong cây** (`n` lớn tới `10^5`): không duyệt bitmask được ⇒ DP trên cây `dp[v][parity]`.
4. **Đồ thị con cảm sinh là cây / có chu trình**: kiểm tra thêm số cạnh cảm sinh `= |s| - 1`.
5. Liên quan: *Count Subtrees With Max Distance Between Cities* (LC 1617) — cũng `n ≤ 15`, duyệt bitmask + kiểm tra liên thông.

In [24]:
# Biến thể 3: cây với n lớn — DP trên cây đếm số cây con liên thông có tổng chẵn
def tree_even_subtrees(nums, edges):
    import sys
    n = len(nums)
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    order, parent = [], [-1] * n
    seen = [False] * n; stack = [0]; seen[0] = True
    while stack:
        u = stack.pop(); order.append(u)
        for w in g[u]:
            if not seen[w]:
                seen[w] = True; parent[w] = u; stack.append(w)
    dp = [[0, 0] for _ in range(n)]      # dp[v][p]: số tập liên thông có gốc cao nhất v, tổng chẵn lẻ p
    total_even = 0
    for v in reversed(order):
        cur = [0, 0]; cur[nums[v] & 1] = 1
        for w in g[v]:
            if w == parent[v]: continue
            nxt = [cur[0], cur[1]]           # không lấy nhánh w
            for a in range(2):
                for b in range(2):
                    nxt[(a + b) & 1] += cur[a] * dp[w][b]
            cur = nxt
        dp[v] = cur
        total_even += cur[0]
    return total_even

rng = random.Random(15)
for _ in range(200):
    n = rng.randint(1, 11)
    edges = [[rng.randrange(i), i] for i in range(1, n)]
    nums = [rng.randint(0, 1) for _ in range(n)]
    assert tree_even_subtrees(nums, edges) == Solution().evenSumSubgraphs(nums, edges)
print("Biến thể 3 (DP trên cây) khớp lời giải bitmask.")

Biến thể 3 (DP trên cây) khớp lời giải bitmask.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tập con + DSU | Dựng đồ thị cảm sinh từng tập | `O(2^n (n+m))` | `O(n+m)` | Dễ viết, dễ đúng |
| 2. Tập con + loang bit | `adj` bitmask, `& sub` | `O(2^n n)` | `O(n)` | **Lời giải chuẩn** |
| 3. DP `conn[mask]` | Bỏ một đỉnh không phải đỉnh cắt | `O(2^n n)` | `O(2^n)` | Khi cần `conn` cho nhiều truy vấn |
| 4. Liệt kê tập liên thông | Quay lui + tập cấm | `O(C n)` | `O(n)` | Đồ thị thưa, chỉ cần tập liên thông |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [0, 1, 1]`, `edges = [[0,1],[1,2],[0,2]]` (tam giác). Đáp án?

**Bài 2 (tính tay).** Đồ thị sao: tâm `0` nối với `1, 2, 3`; `nums = [1, 1, 0, 1]`. Liệt kê và đếm.

**Bài 3 (điền TODO).** Viết `count_connected(n, edges)` đếm **mọi** tập liên thông (không quan tâm tổng).

**Bài 4 (tự viết lại).** Viết lại lời giải bằng DFS đệ quy trên danh sách kề thường (không bitmask cho kề), nhưng vẫn dùng bitmask cho tập.

**Bài 5 (mini challenge).** Giá trị `nums[i]` là số nguyên bất kỳ trong `[0, 100]`; đếm tập liên thông có tổng chia hết cho `3`.

In [25]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_connected(n, edges):
    # TODO: duyệt sub = 1..2^n-1, loang trong sub, đếm khi phủ hết
    pass

In [26]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_div3(nums, edges):
    # TODO: tính sum[mask] bằng DP theo bit thấp nhất, kiểm tra liên thông như Cách 2
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tam giác: mọi tập khác rỗng đều liên thông. (2) Đếm tập có số đỉnh giá trị 1 chẵn: `{0}`, `{1,2}`, `{0,1,2}`. (3) Đáp án `3`.

**Bài 2.** (1) Tập liên thông không chứa tâm chỉ là tập 1 đỉnh lá. (2) Tập chứa tâm: tâm + tập con bất kỳ của lá. (3) Đếm chẵn: `{2}` và các tập `{0} ∪ L` với `1 + #(lá giá trị 1 trong L)` chẵn → tổng cộng `5`.

**Bài 3.** (1) Bỏ điều kiện chẵn lẻ khỏi Cách 2. (2) Đường `n` đỉnh cho `n(n+1)/2`. (3) Đồ thị đầy đủ cho `2^n - 1`.

**Bài 4.** (1) `g[u]` là list. (2) DFS chỉ thăm `w` nếu `sub >> w & 1`. (3) Dùng mảng `vis` mới cho mỗi tập.

**Bài 5.** (1) `s[mask] = s[mask ^ low] + nums[idx(low)]`. (2) Kiểm tra `s[mask] % 3 == 0`. (3) Đối chiếu brute.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2
assert Solution().evenSumSubgraphs([0, 1, 1], [[0, 1], [1, 2], [0, 2]]) == 3
assert Solution().evenSumSubgraphs([1, 1, 0, 1], [[0, 1], [0, 2], [0, 3]]) == 5 == brute([1, 1, 0, 1], [[0, 1], [0, 2], [0, 3]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [28]:
# Bài 3
def count_connected_sol(n, edges):
    adj = build_adj_mask(n, edges)
    return sum(1 for s in range(1, 1 << n) if reach_within(adj, s, s & -s) == s)

assert count_connected_sol(5, [[i, i + 1] for i in range(4)]) == 15
assert count_connected_sol(5, [[u, v] for u in range(5) for v in range(u + 1, 5)]) == 31
print("Bài 3 OK")

Bài 3 OK


In [29]:
# Bài 4
def sol_dfs_list(nums, edges):
    n = len(nums)
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    ans = 0
    for sub in range(1, 1 << n):
        if sum(nums[i] for i in range(n) if sub >> i & 1) % 2:
            continue
        start = (sub & -sub).bit_length() - 1
        vis = [False] * n; vis[start] = True; stack = [start]; cnt = 1
        while stack:
            u = stack.pop()
            for w in g[u]:
                if sub >> w & 1 and not vis[w]:
                    vis[w] = True; cnt += 1; stack.append(w)
        if cnt == bin(sub).count('1'):
            ans += 1
    return ans

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 8); nums = [rng.randint(0, 1) for _ in range(n)]
    edges = [[u, v] for u in range(n) for v in range(u + 1, n) if rng.random() < 0.4]
    assert sol_dfs_list(nums, edges) == Solution().evenSumSubgraphs(nums, edges)
print("Bài 4 OK")

Bài 4 OK


In [30]:
# Bài 5
def count_div3_sol(nums, edges):
    n = len(nums); adj = build_adj_mask(n, edges)
    s = [0] * (1 << n); ans = 0
    for mask in range(1, 1 << n):
        low = mask & -mask
        s[mask] = s[mask ^ low] + nums[low.bit_length() - 1]
        if s[mask] % 3 == 0 and reach_within(adj, mask, low) == mask:
            ans += 1
    return ans

def brute_div3(nums, edges):
    n = len(nums); ans = 0
    for mask in range(1, 1 << n):
        verts = [i for i in range(n) if mask >> i & 1]
        if sum(nums[i] for i in verts) % 3: continue
        se = [(u, v) for u, v in edges if mask >> u & 1 and mask >> v & 1]
        if dsu_components(verts, se) == 1: ans += 1
    return ans

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 8); nums = [rng.randint(0, 100) for _ in range(n)]
    edges = [[u, v] for u in range(n) for v in range(u + 1, n) if rng.random() < 0.4]
    assert count_div3_sol(nums, edges) == brute_div3(nums, edges)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
adj = [0]*n
for u, v in edges: adj[u] |= 1<<v; adj[v] |= 1<<u
ones = bitmask các đỉnh giá trị 1
for sub in range(1, 1<<n):
    if popcount(sub & ones) & 1: continue
    seen = todo = sub & -sub
    while todo:
        b = todo & -todo; todo ^= b
        new = adj[b.bit_length()-1] & sub & ~seen
        seen |= new; todo |= new
    ans += seen == sub
```

**Checklist:**

- [ ] Duyệt `sub` từ `1` (bỏ tập rỗng)?
- [ ] Loang chỉ trong `sub` (`& sub`) — đồ thị **cảm sinh**?
- [ ] Mỗi tập khởi tạo lại `seen`/`todo`?
- [ ] Chẵn lẻ tính bằng `popcount(sub & ones)`?
- [ ] Thêm cạnh **hai chiều** vào `adj`?